### Manual: 04-pinned-wheels

Run this cell in the Kaggle notebook UI. Environment persists across cells in the same session (scripts in `/kaggle/working/smoke_test`, venv, repo clone under `/root/bevfusion_smoke`).


In [ ]:
import os, sys
WORK = os.environ.get('SMOKE_WORK', '/kaggle/working/smoke_test')
if not os.path.exists(os.path.join(WORK, 'smoke_common.py')):
    raise SystemExit('Run 01-bootstrap first in this session (or run the full notebook once to create scripts).')
sys.path.insert(0, WORK)
print('guard ok:', WORK)


In [ ]:
"""Install the pinned stack into the Python 3.8 venv.

torch 1.10.1+cu113 (PyTorch cu113 index), mmcv-full 1.4.0 (OpenMMLab cu113 /
torch1.10.0 wheel index), mmdet 2.20.0 (PyPI). Nothing is upgraded to fit a
newer CUDA: if these wheels cannot be installed the ticket stops here.

Idempotent: skips the install when the venv already reports the pinned versions.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

log = os.path.join(sc.LOGS, "pinned_wheels.log")
uv = "uv pip install --python " + sc.PY

probe = sc.capture(
    '"%s" -c "import torch, mmcv, mmdet; print(torch.__version__, mmcv.__version__, mmdet.__version__)"'
    % sc.PY
)
report = {"probe_before": probe}
already = probe.split() == ["1.10.1+cu113", "1.4.0", "2.20.0"]

commands = [
    uv + ' "setuptools==59.5.0" "wheel" "packaging==21.3" "ninja"',
    uv
    + " --index-url https://download.pytorch.org/whl/cu113"
    + " --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match"
    + ' "torch==1.10.1+cu113" "torchvision==0.11.2+cu113" "numpy==1.22.4"',
    uv
    + " -f https://download.openmmlab.com/mmcv/dist/cu113/torch1.10.0/index.html"
    + ' "mmcv-full==1.4.0"',
    uv
    + ' "mmdet==2.20.0" "matplotlib" "terminaltables" "pycocotools" "six" "scipy"'
    + ' "pyyaml" "numpy==1.22.4" "numba==0.58.1"',
]

report["skipped"] = already
if not already:
    report["steps"] = []
    for command in commands:
        rc, _ = sc.sh(command, log=log, timeout=3600)
        report["steps"].append({"command": command, "rc": rc})
        if rc != 0:
            print("STEP FAILED (rc=%d): %s" % (rc, command))
            print("see " + log)
            break

report["package_list"] = sc.capture('"%s" -m pip list 2>/dev/null || uv pip list --python "%s"' % (sc.PY, sc.PY))
rc, text = sc.run_script("step_env.py", log_name="step_env.log")
report["env_check_rc"] = rc
report["env_check_ok"] = "ENV_OK" in text
sc.save_json("pinned_wheels.json", report)
sc.update(pinned_wheels={"ok": report["env_check_ok"], "skipped": already})
print()
print("PACKAGE_LIST:")
print(report["package_list"])
print("PINNED_WHEELS_OK" if report["env_check_ok"] else "PINNED_WHEELS_FAILED (see %s)" % log)
